# einsum-contraction — worked example 2: Weighted centroid via index contraction

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `einsum-contraction`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When an index appears on two operands but is absent from the output, einsum sums over it. A weighted sum of feature vectors is exactly this: the sample axis is shared by the features and the weights and is contracted away, leaving only the feature dimension.

## Worked solution

Given `feats: (N, D)` (one feature vector per sample) and `weights: (N,)` (a scalar weight per sample), we want the weighted sum over samples, shape `(D,)`.

1. Look at the indices. `n` (the sample axis) appears on both operands but must NOT appear in the output — so it is contracted. `d` (the feature axis) appears only on `feats` and on the output, so it is preserved.
2. The pattern is `'n d, n -> d'`. The weights operand has only the `n` label because it is a 1-D vector indexed by sample.
3. For each output position `d`, einsum computes `sum_n feats[n,d] * weights[n]` — every feature dimension gets the same per-sample weights applied, then summed across samples.
4. The result `(D,)` equals `(feats * weights[:, None]).sum(0)`, which we check.

In [ ]:
import torch as t
import einops

t.manual_seed(1)
feats = t.randn(6, 3)
weights = t.rand(6)

def weighted_centroid(feats, weights):
    return einops.einsum(feats, weights, 'n d, n -> d')

out = weighted_centroid(feats, weights)
print(out.shape)
print('matches manual:', bool(t.allclose(out, (feats * weights[:, None]).sum(0), atol=1e-5)))